# Qwen3-VL-4B Test: Next-Gen Vision Model

This notebook tests **Qwen3-VL-4B** via the LM Studio local server using **Streaming Mode**.

### 1. Setup

```bash
uv pip install openai pillow
```


In [1]:
import base64
import time
from openai import OpenAI
from PIL import Image
import io
import sys

# Point to the local server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


img_path = "ollama_test_table.png"
base64_image = encode_image(img_path)

print("Setup complete. Ready to talk to Qwen3-VL.")

Setup complete. Ready to talk to Qwen3-VL.


### 2. Run Inference: Precision Document Extraction (STREAMING)

We are testing if the 4B version of Qwen3 handles Arabic legal text better than the 2.5-7B version.


In [2]:
print("Requesting streaming inference from Qwen3-VL-4B...\n")
start_time = time.time()

response = client.chat.completions.create(
    model="qwen3-vl-4b-instruct",  # LM Studio usually uses the currently loaded model
    messages=[
        {
            "role": "system",
            "content": "You are an expert Arabic OCR and document analyst. Your goal is to transcribe the provided image into Markdown perfectly. Transcribe all titles, text, and tables exactly as they appear. For tables, use the Markdown pipe format (| column |). Pay extreme attention to Arabic spelling (e.g., ensure 'الجنحة' is spelled correctly with a 'ح').",
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Please extract all the text and tables from this document in Markdown format.",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                },
            ],
        },
    ],
    max_tokens=4096,
    temperature=0,
    stream=True,
)

full_response = ""
print("--- Real-time Output ---")
for chunk in response:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        full_response += content

end_time = time.time()
print(f"\n\n--- Inference Finished in {end_time - start_time:.2f} seconds ---")

Requesting streaming inference from Qwen3-VL-4B...

--- Real-time Output ---
```markdown
# «الجَنْح»

| النقطة الواجب خصِمها | الجُنحة | الرقم التّرنيهي |
|---|---|---|
| ... | ... | 01 |
| ... | ... | ... |
| 6 | مسافة مركبة تحت تأثير الكحول<br>أو تحت تأثير المواد المخدرة<br>- رفض الخضوع للرّاتز المشار<br>إليه في المادة 207 أدناه أو للتحققات<br>أو لاختبارات الكشف المنصوص عليها<br>في المادتين 208 و 213 أدناء | 07 |
| ... | ... | ... |
| 2 | السائق الذي وجه إليه الأمر<br>بالتوقف وامتنع من تنفيذه أو لم يحترم<br>الأمر بتوفيف المركبة أو رفض مسافة<br>مركّبته أو العمل على مسابقتها إلى المحجز<br>أو رفض الامتثال للأوامر القانونية<br>الصادرة إليه | 13 |
| ... | ... | ... |
| ... | ... | 18 |

# «المُخَالِفَات»

| النقطة الواجب خصِمها | المخالفات | الرقم التّرنيهي |
|---|---|---|
| ... | ... | 19 |
| 1 | عدم احترام إجبارية استعمال<br>حزام السلامة. | 30 |
| ... | ... | 31 |
| 1 | -الاستعمال أو التحدث بالهاتف<br>ممسوكة باليد أثناء السياقة أو أي جهاز<br>أُخْرِي قوم بوظائف الهاتف. | 32 |
```

--- In

### 3. Display Rendered Result


In [3]:
from IPython.display import display, Markdown

display(Markdown(full_response))

```markdown
# «الجَنْح»

| النقطة الواجب خصِمها | الجُنحة | الرقم التّرنيهي |
|---|---|---|
| ... | ... | 01 |
| ... | ... | ... |
| 6 | مسافة مركبة تحت تأثير الكحول<br>أو تحت تأثير المواد المخدرة<br>- رفض الخضوع للرّاتز المشار<br>إليه في المادة 207 أدناه أو للتحققات<br>أو لاختبارات الكشف المنصوص عليها<br>في المادتين 208 و 213 أدناء | 07 |
| ... | ... | ... |
| 2 | السائق الذي وجه إليه الأمر<br>بالتوقف وامتنع من تنفيذه أو لم يحترم<br>الأمر بتوفيف المركبة أو رفض مسافة<br>مركّبته أو العمل على مسابقتها إلى المحجز<br>أو رفض الامتثال للأوامر القانونية<br>الصادرة إليه | 13 |
| ... | ... | ... |
| ... | ... | 18 |

# «المُخَالِفَات»

| النقطة الواجب خصِمها | المخالفات | الرقم التّرنيهي |
|---|---|---|
| ... | ... | 19 |
| 1 | عدم احترام إجبارية استعمال<br>حزام السلامة. | 30 |
| ... | ... | 31 |
| 1 | -الاستعمال أو التحدث بالهاتف<br>ممسوكة باليد أثناء السياقة أو أي جهاز<br>أُخْرِي قوم بوظائف الهاتف. | 32 |
```